In [1]:

import datetime
import numpy as np
import cv2
from itertools import cycle
import pickle
import tqdm
from matplotlib import pyplot as plt
import scipy.signal as sig
import pandas as pd
from scipy.stats import kde
from eye_tracking_system_tools.preprocessing.BlockSync_class import BlockSync
from eye_tracking_system_tools.preprocessing import utility_functions as uf
from scipy import signal
import bokeh
import seaborn as sns
from matplotlib import rcParams
from pathlib import Path
%matplotlib inline
plt.style.use('default')
rcParams['pdf.fonttype'] = 42  # Ensure fonts are embedded and editable
rcParams['ps.fonttype'] = 42  # Ensure compatibility with vector outputs

In [ ]:
def create_distance_plot(distances, top_dist_to_show=500):
    # Create cumulative distribution plot
    sns.set(style="whitegrid")
    fig, axs = plt.subplots(2, figsize=(6, 6), dpi=150)
    
    axs[0].set_title('Cumulative Euclidean Distances for Camera Jitter', fontsize=15)
    axs[0].set_ylabel('Cumulative \n % of Frames')
    axs[0].set_xlim(0, top_dist_to_show)
    axs[0].grid(False)
    
    # Create histogram and cumulative distribution
    sns.kdeplot(distances, cumulative=True, label='Left Eye', ax=axs[0], linewidth=4, c='black')
    
    axs[1].hist(distances, bins=np.linspace(0, top_dist_to_show, 20), log=False, color='black')

    # Set title and labels
    title = 'Image displacement histogram'
    axs[1].set_title(title, fontsize=15)
    axs[1].set_xlabel('Euclidean Displacement [$\mu$m]', fontsize=15)
    axs[1].set_xscale('linear')
    axs[1].set_yscale('linear')
    axs[1].set_ylabel('Frame count', fontsize=15)

    # Adjust tick label sizes
    axs[1].tick_params(axis='x', which='major', labelsize=15)

    # Set white background and black text
    axs[1].set_facecolor('white')
    axs[1].title.set_color('black')
    axs[1].xaxis.label.set_color('black')
    axs[1].yaxis.label.set_color('black')
    axs[1].tick_params(colors='black')
    axs[1].grid(False)

    plt.tight_layout()

    return fig, axs
def add_intermediate_elements(input_vector, gap_to_bridge):
    # Step 1: Calculate differences between each element
    differences = np.diff(input_vector)

    # Step 2: Add intervening elements based on the diff_threshold
    output_vector = [input_vector[0]]
    for i, diff in enumerate(differences):
        if diff < gap_to_bridge:
            # Add intervening elements
            output_vector.extend(range(input_vector[i] + 1, input_vector[i + 1]))

        # Add the next element from the original vector
        output_vector.append(input_vector[i + 1])

    return np.sort(np.unique(output_vector))

def find_jittery_frames(block, eye, max_distance, diff_threshold, gap_to_bridge=6):
    
    #input checks
    if eye not in ['left', 'right']:
        print(f'eye can only be left/right, your input: {eye}')
        return None
    # eye setup
    if eye == 'left':
        jitter_dict = block.le_jitter_dict
        eye_frame_col = 'L_eye_frame'
    elif eye == 'right':
        jitter_dict = block.re_jitter_dict
        eye_frame_col = 'R_eye_frame'
    
    df_dict = {'left':block.le_df,
               'right':block.re_df}
    
    df = pd.DataFrame.from_dict(jitter_dict)
    indices_of_highest_drift = df.query("top_correlation_dist > @max_distance").index.values
    diff_vec = np.diff(df['top_correlation_dist'].values)
    diff_peaks_indices = np.where(diff_vec > diff_threshold)[0]
    video_indices = np.concatenate((diff_peaks_indices, indices_of_highest_drift))
    print(f'the diff based jitter frame exclusion gives: {np.shape(diff_peaks_indices)}')
    print(f'the threshold based jitter frame exclusion gives: {np.shape(indices_of_highest_drift)}')
    
    # creates a bridged version of the overly jittery frames (to contend with single frame outliers)
    video_indices = add_intermediate_elements(video_indices, gap_to_bridge=gap_to_bridge)
    # This is the input you should give to the BlockSync.remove_eye_datapoints function (which already maps it to the df) 
    
    
    # translates the video indices to le/re dataframe rows
    df_indices_to_remove = df_dict[eye].loc[df_dict[eye][eye_frame_col].isin(video_indices)].index.values
    
    return df_indices_to_remove, video_indices

def bokeh_plotter(data_list, label_list,
                  plot_name='default',
                  x_axis='X', y_axis='Y',
                  peaks=None, peaks_list=False, export_path=False):
    """Generates an interactive Bokeh plot for the given data vector.
    Args:
        data_list (list or array): The data to be plotted.
        label_list (list of str): The labels of the data vectors
        plot_name (str, optional): The title of the plot. Defaults to 'default'.
        x_axis (str, optional): The label for the x-axis. Defaults to 'X'.
        y_axis (str, optional): The label for the y-axis. Defaults to 'Y'.
        peaks (list or array, optional): Indices of peaks to highlight on the plot. Defaults to None.
        export_path (False or str): when set to str, will output the resulting html fig
    """
    color_cycle = cycle(bokeh.palettes.Category10_10)
    fig = bokeh.plotting.figure(title=f'bokeh explorer: {plot_name}',
                                x_axis_label=x_axis,
                                y_axis_label=y_axis,
                                plot_width=1500,
                                plot_height=700)

    for i, vec in enumerate(range(len(data_list))):
        color = next(color_cycle)
        data_vector = data_list[vec]
        if label_list is None:
            fig.line(range(len(data_vector)), data_vector, line_color=color, legend_label=f"Line {len(fig.renderers)}")
        elif len(label_list) == len(data_list):
            fig.line(range(len(data_vector)), data_vector, line_color=color, legend_label=f"{label_list[i]}")
        if peaks is not None and peaks_list is True:
            fig.circle(peaks[i], data_vector[peaks[i]], size=10, color=color)

    if peaks is not None and peaks_list is False:
        fig.circle(peaks, data_vector[peaks], size=10, color='red')

    if export_path is not False:
        print(f'exporting to {export_path}')
        bokeh.io.output.output_file(filename=str(export_path / f'{plot_name}.html'), title=f'{plot_name}')
    bokeh.plotting.show(fig)   

    
def get_frame_count(video_path):
        """
        Get the number of frames for the video in the specified path using OpenCV.
    
        Parameters:
            video_path (str): Path to the video file.
    
        Returns:
            int: Number of frames in the video.
        """
        
        # Open the video file
        cap = cv2.VideoCapture(video_path)
    
        # Check if the video file is opened successfully
        if not cap.isOpened():
            print("Error: Could not open the video file.")
            return -1
    
        # Get the total number of frames in the video
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
        # Release the VideoCapture object
        cap.release()
    
        return frame_count


In [ ]:

# define a single block to synchronize and finally export l/r_eye_data csv files:
# this step creates block_collection - a list of BlockSync objects of interest
block_numbers = [15]
bad_blocks = [] #
experiment_path = Path(r"D:\sample_data_for_eye_repo")
animal = 'PV_106'

block_collection = uf.block_generator(block_numbers=block_numbers,
                                      experiment_path=experiment_path,
                                      animal=animal,
                                      bad_blocks=bad_blocks,regev=True)
for block in block_collection:
    block.channeldict = None
    if block.animal_call == 'PV_208':
        block.channeldict={1: 'LED_driver',
                           7: 'L_eye_TTL',
                           2: 'Arena_TTL',
                           8: 'R_eye_TTL'}
# create a block_dict object for ease of access:
block_dict = {}
for b in block_collection:
    block_dict[str(b.block_num)] = b
block = block_collection[0]

In [ ]:
block = block_collection[0]

In [ ]:
block.block_get_lizard_movement()

In [ ]:
# over here, perform a rolling window analysis and select threshold for active/quite segmentation
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import pandas as pd
def rolling_window_analysis(df, window_size=10000, step_size=1000):
    """
    Perform a rolling window analysis on the movAll column of the input dataframe.

    Args:
        df (pd.DataFrame): Input dataframe with 't_mov_ms' (timestamps in ms) and 'movAll' (magnitudes).
        window_size (int): Size of the rolling window in milliseconds (default 10,000 ms).
        step_size (int): Step size for the rolling window in milliseconds (default 1,000 ms).

    Returns:
        pd.DataFrame: A dataframe containing the start time of each window and the average movAll value.
    """
    # Ensure the dataframe is sorted by time
    df = df.sort_values('t_mov_ms').reset_index(drop=True)
    
    # Get the range of timestamps
    t_min = df['t_mov_ms'].min()
    t_max = df['t_mov_ms'].max()
    
    # Create the rolling window start times
    window_starts = np.arange(t_min, t_max + step_size, step_size)
    
    # Initialize results
    results = {'window_start': [], 'average_movAll': []}
    
    for start in window_starts:
        # Define the window range
        end = start + window_size
        # Filter events within the window
        window_data = df[(df['t_mov_ms'] >= start) & (df['t_mov_ms'] < end)]
        # Compute the average movAll or set to 0 if no events occurred
        avg_movAll = window_data['movAll'].mean() if not window_data.empty else 0
        # Append results
        results['window_start'].append(start)
        results['average_movAll'].append(avg_movAll)
    
    # Return the results as a dataframe
    return pd.DataFrame(results)

df =  rolling_window_analysis(block.liz_mov_df,10000,1000)
# Initialize Plotly figure
fig = go.Figure()

# Add the movement trace
fig.add_trace(go.Scatter(
    x=df['window_start'],
    y=df['average_movAll'],
    mode='lines+markers',
    name='Average Movement',
    line=dict(color='blue')
))

# Add an initial threshold line
initial_threshold = 0.5
fig.add_trace(go.Scatter(
    x=df['window_start'],
    y=[initial_threshold] * len(df),
    mode='lines',
    name='Threshold',
    line=dict(color='red', dash='dash')
))

# Update layout for interactivity
fig.update_layout(
    title="Interactive Movement Threshold Selection",
    xaxis_title="Time (ms)",
    yaxis_title="Average Movement",
    sliders=[{
        "active": 5,
        "currentvalue": {"prefix": "Threshold: "},
        "steps": [
            {"label": str(round(threshold, 2)), "method": "update", 
             "args": [{"y": [df['average_movAll'], [threshold] * len(df)]}]}
            for threshold in [x / 10.0 for x in range(0, 20)]
        ]
    }]
)

# Show the interactive plot
fig.show()

In [ ]:
threshold = 0.08
df['behavior'] = df['average_movAll'].apply(lambda x: 'active' if x > threshold else 'quiet')
def create_behavior_df(df):
    """
    Transform the annotated dataframe into a compact behavioral dataframe.
    
    Args:
        df (pd.DataFrame): Dataframe with 'window_start' and 'behavior' columns.

    Returns:
        pd.DataFrame: Compact behavioral dataframe with 'start_time', 'end_time', and 'annotation'.
    """
    # Initialize variables
    behavior_df = []
    current_behavior = df['behavior'].iloc[0]
    start_time = df['window_start'].iloc[0]

    for i in range(1, len(df)):
        # If the behavior changes, mark the end of the current behavioral window
        if df['behavior'].iloc[i] != current_behavior:
            end_time = df['window_start'].iloc[i]  # End time is the start of the next window
            behavior_df.append({
                'start_time': start_time,
                'end_time': end_time,
                'annotation': current_behavior
            })
            # Start a new behavioral window
            current_behavior = df['behavior'].iloc[i]
            start_time = df['window_start'].iloc[i]
    
    # Append the last behavioral window
    end_time = df['window_start'].iloc[-1] + 1000  # Include the last second
    behavior_df.append({
        'start_time': start_time,
        'end_time': end_time,
        'annotation': current_behavior
    })
    
    # Convert to a dataframe
    return pd.DataFrame(behavior_df)

# Apply to your dataframe
behavior_df = create_behavior_df(df)
print(behavior_df)



In [ ]:
block.behavior_state = behavior_df
csv_path = block.analysis_path / f"block_{block.block_num}_behavior_state.csv"
behavior_df.to_csv(csv_path, index=False)
print(f"Behavior state saved to {csv_path}")